# Пайплайн от сценария до метрик

Цепочка повторяет схему «физика → модель наблюдения → прогноз»:

```
сценарий → симуляция → наблюдение → признаки → окна → модель → потери → обучение → метрики
```

Каждая стадия здесь — вызов функции или запись реестра из `src/`. Ноутбук ничего не пишет в `experiments/results/` и в кэш траекторий (`cache="none"`), полный прогон на CPU занимает около двух минут.

## 0. Воспроизводимость

Всё случайное в пайплайне задаётся сидами: начальные состояния траекторий берутся из глобального генератора NumPy, веса и перемешивание — из torch. Один поток torch даёт на одной машине одни и те же числа от запуска к запуску. `environment_info()` — то, что записывается в `*.provenance.json` рядом с каждым CSV.

In [ ]:
import dataclasses
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src import pipeline, reproducibility

reproducibility.set_num_threads(1)
reproducibility.environment_info()

## 1. Сценарии

Пять сценариев A–E: один кубит с постоянным и с зависящим от времени γ(t), с релаксацией σ₋ и дефазировкой σ_z, и два кубита с взаимодействием XXZ или TFIM. Параметры каждой системы (частота, J, форма γ(t)) разыгрываются из одного генератора, поэтому конфигурации воспроизводятся по сиду. Конфигурация — это только описание системы, ничего ещё не посчитано.

In [ ]:
from src.data import scenarios

pd.Series(scenarios.SCENARIOS, name="сценарий")

In [ ]:
groups = scenarios.build_configs(n_per_scenario=3, seed=0)
pd.DataFrame([
    {"сценарий": label,
     "кубиты": c.n_qubits.value,
     "взаимодействие": c.interaction_type.value if c.n_qubits.value == 2 else "-",
     "диссипатор": c.dissipator.operator_type.value,
     "omega": round(c.omega, 2),
     "J": round(c.J, 2) if c.n_qubits.value == 2 else "-",
     "t_max": c.t_max,
     "критерий T2": c.decoherence_criterion.value}
    for label in "ABCDE" for c in groups[label][:2]
])

## 2. Симуляция

Симулятор решает уравнение Линдблада и возвращает траекторию: точные ожидаемые значения наблюдаемых на сетке времени и время декогеренции T2 — первое пересечение когерентностью C(t) уровня 1/e от начального значения. Вариант симулятора выбирается по псевдониму из реестра: `plain` для публикационной физики, `entanglement` дополнительно пишет парные корреляторы и конкурренс.

In [ ]:
from src.simulation import registry as simulators

plain = simulators.build_simulator("plain")
entangled = simulators.build_simulator("entanglement")

trajectories = {}
for label in "ABCDE":
    reproducibility.seed_numpy_global(0)
    simulator = entangled if label in "DE" else plain
    trajectories[label] = simulator.simulate(groups[label][0])

pd.Series({label: t.t_decoh for label, t in trajectories.items()}, name="T2")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 2.8))
for ax, (label, t) in zip(axes, trajectories.items()):
    coherence = t.observables["coherence_l1"]
    ax.plot(t.times, coherence)
    ax.axhline(t.system_config.decoherence_threshold * coherence[0],
               ls="--", c="gray", lw=1)
    ax.axvline(t.t_decoh, c="C3", lw=1)
    ax.set(title=f"{label}: T2 = {t.t_decoh:.2f}", xlabel="t")
axes[0].set_ylabel("C(t)")
plt.show()

Что считать временем декогеренции — выбор исследователя, и от него зависит цель прогноза. Для двухкубитной системы одна и та же траектория (то же начальное состояние) даёт три разных T2: по ℓ₁-когерентности в вычислительном базисе, по ней же в базисе собственных векторов H и по моменту внезапной смерти запутанности (первый нуль конкурренса).

In [ ]:
from src.domain import value_objects

criteria = ("coherence", "coherence_energy", "entanglement_death")
by_criterion = {}
for name in criteria:
    changed = dataclasses.replace(
        groups["D"][0],
        decoherence_criterion=value_objects.DecoherenceCriterion(name))
    reproducibility.seed_numpy_global(0)
    by_criterion[name] = entangled.simulate(changed)

pd.DataFrame({name: {"T2": t.t_decoh, "censored": t.censored}
              for name, t in by_criterion.items()}).T

In [ ]:
t = by_criterion["coherence"]
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(t.times, t.observables["coherence_l1"], label="C(t), ℓ₁-когерентность")
ax.plot(t.times, t.observables["concurrence"], label="конкурренс")
for (name, found), colour in zip(by_criterion.items(), ("C2", "C1", "C3")):
    ax.axvline(found.t_decoh, ls="--", c=colour, label=f"T2: {name}")
ax.set(xlabel="t")
ax.legend(fontsize=8)
plt.show()

## 3. Модель наблюдения

Симулятор отдаёт точные средние. Модель наблюдения решает, что из них видит предсказатель. Здесь живёт вопрос научного руководителя: какие данные реально доступны в эксперименте.

**Что сейчас подаётся на вход.** Входы модели — это *средние по ансамблю*: ожидаемые значения ⟨σ⟩, чистота и когерентность, вычисленные по матрице плотности, то есть усреднение по бесконечному числу повторных приготовлений одного и того же состояния. Реализованы три модели наблюдения:

- `exact` — точные средние без шума;
- `gaussian` — независимый гауссов шум σ в каждой точке каждого окна;
- `ar1` — шум, коррелированный по времени (AR(1) с коэффициентом φ).

In [ ]:
from src.data import observation, windows

one_window = windows.build_dataset(
    [trajectories["B"]], window_length=20, horizon=1.0,
    samples_per_trajectory=1, train_ratio=0.0, val_ratio=0.0,
    min_window_gap=0, include_censored=False, seed=0,
    rng=np.random.default_rng(0))
column = one_window.feature_names.index("coherence_l1")

settings = {"exact": {}, "gaussian": {"sigma": 0.05},
            "ar1": {"sigma": 0.05, "phi": 0.9}}
fig, ax = plt.subplots(figsize=(6.5, 3))
for name, kwargs in settings.items():
    observed = observation.OBSERVATIONS[name](
        one_window, rng=np.random.default_rng(0), **kwargs)
    ax.plot(observed.sequences[0, :, column], label=name)
ax.set(xlabel="шаг окна", ylabel="C(t) в окне")
ax.legend()
plt.show()

**Запланировано, не реализовано** (в реестре `OBSERVATIONS` этих записей пока нет, кода тоже):

- `"shots"` — конечное число повторений: каждое среднее оценивается по N проективным измерениям, шум биномиальный; когерентность и чистота считаются из одних и тех же оценок, а не независимо;
- `"sequential"` — запись одиночных измерений одного экземпляра системы с обратным влиянием измерения на состояние (cf. Luchnikov et al., PRL 2020);
- набор признаков `"local"` — только шесть локальных средних Паули (без парных корреляторов и без чистоты).

Пока этого нет, результаты относятся к ансамблевым средним с аддитивным шумом, и так их нужно читать.

## 4. Признаки

Какие столбцы окна видит модель, задаёт набор признаков. Для двух кубитов `base` — восемь базовых наблюдаемых (шесть локальных средних Паули, чистота, когерентность), `corr` добавляет девять парных корреляторов, `corr_conc` — ещё и конкурренс. Все наборы режутся из одной симуляции, поэтому сравнение идёт на одинаковых окнах.

In [ ]:
from src.data import features

two_qubit = windows.build_dataset(
    [trajectories["D"], trajectories["E"]], window_length=20, horizon=1.0,
    samples_per_trajectory=3, train_ratio=0.5, val_ratio=0.0,
    min_window_gap=0, include_censored=False, seed=0,
    rng=np.random.default_rng(0))
pd.Series({name: len(features.select(two_qubit, name).feature_names)
           for name in features.FEATURE_SETS}, name="столбцов в окне")

## 5. Окна и датасет

Из каждой траектории вырезаются окна фиксированной длины, которые заканчиваются до T2. Метка окна — оставшееся время T2 − t_obs и признак риска «T2 наступит в пределах горизонта». Разбиение на train/val/test делается по траекториям, а не по окнам, чтобы окна одной траектории не попали в разные части. Всё это делает `GenerateDatasetUseCase`; `pipeline.generate` вызывает его по описанию `DataSpec`.

In [ ]:
spec = pipeline.DataSpec(
    name="demo", scenarios=("B",), n_per_scenario=40, window_length=20,
    samples_per_trajectory=5, cache="none")
dataset = pipeline.generate(spec, seed=42, cache={})

{"окон": len(dataset.sequences),
 "форма окна": dataset.sequences.shape[1:],
 "train / val / test": (len(dataset.train_idx), len(dataset.val_idx),
                        len(dataset.test_idx)),
 "признаки": dataset.feature_names}

In [ ]:
from src.application import generate_dataset

use_case = generate_dataset.GenerateDatasetUseCase(plain, store=None)
direct = use_case.execute(generate_dataset.GenerateDatasetCommand(
    configs=pipeline.system_configs(spec, 42), window_length=20,
    horizon=1.0, samples_per_trajectory=5, seed=42))
np.array_equal(direct.sequences, dataset.sequences)

## 6. Модели и функции потерь

Модель и функция потерь выбираются строкой. `physics_only` — формула по наклону ln C(t), без обучения; `physics_lstm` — BiLSTM, которая дообучает поправку к этой формуле; `transformer` — сеть без физического приора. Потеря — регрессионная часть обучения (для T2); часть для риска обучается вместе с ней.

In [ ]:
from src.models import registry as model_registry
from src.training import losses

pd.Series({alias: type(model_registry.build_model(alias)).__name__
           for alias in model_registry.MODELS}, name="модели")

In [ ]:
pd.Series({alias: type(loss).__name__ for alias, loss in losses.LOSSES.items()},
          name="потери")

## 7. Обучение и оценка

`trainer.train` обучает модель на train-части датасета и возвращает её же. `backtest.evaluate` считает метрики на тестовых окнах: MAE, RMSE, R² и MAPE по времени T2 и AUROC по риску; дополнительные метрики из реестра `METRICS` просятся по псевдониму. Здесь три эпохи на сорока траекториях — для проверки хода, а не для качества.

In [ ]:
from src.evaluation import backtest, metrics
from src.training import trainer

model = model_registry.build_model("physics_lstm")
trainer.train(model, dataset, epochs=3, batch_size=32, lr=1e-3, seed=42,
              loss="huber")
scores = backtest.evaluate(
    model, dataset, horizon=1.0, metrics=["r2", "mae", "rmse", "auroc"])
baseline = backtest.evaluate(
    model_registry.build_model("physics_only"), dataset, horizon=1.0,
    metrics=["r2", "mae", "rmse", "auroc"])
pd.DataFrame({"physics_only": baseline.extra, "physics_lstm": scores.extra})

In [ ]:
fig, ax = plt.subplots(figsize=(3.6, 3.4))
ax.scatter(scores.actuals, scores.predictions, s=12)
limit = [0, max(scores.actuals.max(), scores.predictions.max())]
ax.plot(limit, limit, c="gray", lw=1)
ax.set(xlabel="T2", ylabel="прогноз T2", title="physics_lstm, 3 эпохи")
plt.show()
sorted(metrics.METRICS)

## 8. Эксперимент целиком

Ту же цепочку можно описать данными. `ExperimentConfig` называет все выбранные варианты — данные, модель, потери, аугментацию шумом, условия, на которых модель оценивается, — а `run_seeds` исполняет её для каждого сида и возвращает таблицу, одна строка на условие. Ниже R16 в миниатюре: обычное обучение и обучение на копиях окон с шумом, оценка на чистых окнах и на окнах с гауссовым шумом σ = 0.05.

In [ ]:
data = pipeline.DataSpec(
    name="mini", scenarios=("B", "C"), n_per_scenario=20, window_length=20,
    cache="none")
tests = tuple(
    pipeline.EvalSpec("gaussian", sigma=sigma, tags=(("sigma", sigma),))
    for sigma in (0.0, 0.05))
configs = [
    pipeline.ExperimentConfig(
        data=data, model="physics_lstm", epochs=3, tests=tests,
        augmentation=augmentation, tags=(("arm", arm),))
    for arm, augmentation in (
        ("clean", None), ("noise_aware", pipeline.Augmentation(copies=2)))
]
table = pipeline.run_seeds(configs, seeds=[42, 43])
table[["seed", "arm", "sigma", "r2", "mae", "auroc", "n_samples"]]

Сравнение рук парное по сидам: одинаковые данные и одинаковый шум в тесте, различается только обучение. Два сида — только чтобы показать вызовы; выводов из них делать нельзя.

In [ ]:
from src.evaluation import stats

noisy = table[table["sigma"] == 0.05]
paired = stats.paired_difference(
    noisy[noisy["arm"] == "noise_aware"]["r2"].tolist(),
    noisy[noisy["arm"] == "clean"]["r2"].tolist())
print(f"noise_aware - clean, R2 при sigma=0.05: {paired.mean:+.3f} "
      f"± {paired.std:.3f} (n={paired.n}, лучше в {paired.n_positive})")
{arm: stats.mean_std(group["r2"].tolist())
 for arm, group in table.groupby("arm")}

## 9. Опубликованные результаты

CSV в `experiments/results/` — результаты полных прогонов; здесь они только читаются, ничего не записывается. Агрегируются теми же функциями `stats`, что и в сводках экспериментов.

R13: шесть моделей на 400 независимых траекториях на сценарий, три сида.

In [ ]:
RESULTS = ROOT / "experiments" / "results"
r13 = pd.read_csv(RESULTS / "r13_independent_test.csv")
pd.DataFrame([
    {"сценарий": scenario, "модель": variant,
     "R2": f"{mean:.3f} ± {std:.3f}"}
    for (scenario, variant), group in r13.groupby(["scenario", "variant"])
    for mean, std in [stats.mean_std(group["r2"].tolist())]
]).pivot(index="модель", columns="сценарий", values="R2")

In [ ]:
wide = r13.pivot_table(index=["scenario", "seed"], columns="variant",
                       values="r2")
for scenario in "BC":
    diff = stats.paired_difference(
        wide.loc[scenario]["physics_lstm"].tolist(),
        wide.loc[scenario]["lstm_only"].tolist())
    print(f"{scenario}: physics_lstm - lstm_only, R2 {diff.mean:+.4f} "
          f"± {diff.std:.4f} (n={diff.n}, лучше в {diff.n_positive})")

R22: обучение с AR(1)-шумом, оценка при сильном коррелированном шуме (σ = 0.10, φ = 0.9).

In [ ]:
r22 = pd.read_csv(RESULTS / "r22_ar1_noise_training.csv")
hard = r22[(r22["sigma"] == 0.10) & (r22["phi"] == 0.9)]
pd.DataFrame({arm: dict(zip(("R2 mean", "R2 std"),
                            stats.mean_std(group["r2"].tolist())))
              for arm, group in hard.groupby("arm")}).T

R25: кривая обучения для цели «внезапная смерть запутанности» — R² от числа обучающих траекторий для трёх наборов признаков.

In [ ]:
r25 = pd.read_csv(RESULTS / "r25_esd_data_scaling.csv")
pd.DataFrame([
    {"сценарий": scenario, "признаки": feats, "n_traj": n_traj,
     "R2": stats.mean_std(group["r2"].tolist())[0]}
    for (scenario, feats, n_traj), group
    in r25.groupby(["scenario", "features", "n_traj"])
]).pivot(index=["сценарий", "признаки"], columns="n_traj", values="R2").round(3)